In [ ]:
import sys, os
import matplotlib.pyplot as plt

# Project styling
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'src'))
from dimp.utils import init_matplotlib, get_colors
init_matplotlib()
colors = get_colors()

# Import plotting functions from the visualization script
from plot_invpend_dt import (
    load_method_results,
    load_loss_results,
    plot_method_results,
    plot_invpend_trajectory,
    print_continuous_costs,
    plot_baseline_sweep,
    plot_loss_results,
    plot_loss_comparison,
    _build_method_solutions,
    plot_density_analysis,
    plot_cross_correlation_analysis,
    s0, s_goal, A, B, Q, R, T, u_max, x_max, n_s, n_u, e0, STATE_LABELS,
)
import numpy as np

In [ ]:
data_dir = os.path.join("data", "invpend_dt")

# Select which alternative losses to plot (None = all available)
selected_losses = None  # e.g. ["L_IV", "L_FI"]

## Linearized Cart-Pole (Go-to-Goal)

LTI system with dynamics $\dot{s} = A s + B u$.

**States**: $[x, \dot{x}, \theta, \dot{\theta}]$ (position, velocity, angle, angular velocity)

**Input**: $F$ (horizontal force on cart)

**Parameters**:
- $m = 0.2$ kg (pendulum mass), $M = 1.0$ kg (cart mass), $l = 0.5$ m (pendulum length)
- $s_0 = [0, 0, 0.1, 0]$ (perturbed pole)
- $s_{\text{goal}} = [5, 0, 0, 0]$ (go to $x=5$, stop, upright)
- $T = 5$ s, $n = 40$ timesteps
- $Q = \text{diag}(1, 0.1, 10, 0.1)$ (heavy angle penalty)
- $R = 0.01$

**Constraints**:
- $|F| \le 10$ N (input bound)
- $|\dot{x}| \le 2$ m/s (velocity constraint)
- $|\theta| \le 0.15$ rad (angle constraint, $\approx 8.6^\circ$)
- $s_N = s_{\text{goal}}$ (terminal equality)

**Expected sampling pattern**: dense-coarse-dense (stabilize pole, cruise, brake + re-stabilize).

**Error coordinates**: All QPs are formulated in error coordinates $e = s - s_{\text{goal}}$. Since $A s_{\text{goal}} = 0$ (position does not appear in the dynamic equations), both Euler and ZOH dynamics are identical in error coordinates.

## Baseline Sweep

Sweep uniform-dt constrained LQR over $n = 40, 60, \ldots, 200$ to study how the solution quality depends on the number of timesteps.

In [ ]:
plot_baseline_sweep(results_dir=None, show=True, n_tests=[40, 200])

## DQP with Reparametrized Parameters (Rep)

Reparametrize timesteps with the softmax simplex:
$$\Delta t_k = \epsilon + (T - n\epsilon) \frac{e^{\theta_k}}{\sum_j e^{\theta_j}}$$

Euler dynamics with parametric $\Delta t_k$ directly in the QP.

**Loss**: $\mathcal{L} = \sum_{k=0}^{N} \Delta t_k \left(\lVert e_k \rVert^2_Q + \lVert u_k \rVert^2_R \right)$

In [ ]:
method_results = load_method_results(data_dir)

if "rep" in method_results:
    plot_method_results("rep", method_results["rep"], results_dir=None, show=True)

    # Trajectory plot
    sol_rep = method_results["rep"]["sol"]["time_scaled"]
    hist_rep = [h for h in method_results["rep"]["history"] if h['method'] == 'time_scaled']
    dts_rep = np.array(hist_rep[-1]['dts']).flatten()
    n_rep = method_results["rep"]["n"]
    plot_invpend_trajectory(sol_rep, dts_rep, n_rep, title="Rep: time_scaled", show=True)

## DQP with ZOH Exact Discretization and Exact Integrated Cost (ZOH)

Exact ZOH discretization of both dynamics and quadratic stage cost via Van Loan block matrix exponential.

**Dynamics**: $s_{k+1} = A_{d,k} s_k + B_{d,k} u_k$, computed via $\exp\bigl(\begin{bmatrix} A & B \\ 0 & 0 \end{bmatrix} \Delta t_k\bigr)$

**Cost**: $\ell_k = z_k^T \bar{W}_k z_k$ where $z_k = [e_k; u_k]$ and $\bar{W}_k$ is the exact integrated cost matrix.

The matrices $(\bar{W}_k, A_{d,k}, B_{d,k})$ depend nonlinearly on $\Delta t_k$ through `torch.matrix_exp`, but are **fixed parameters** inside the QP.

In [ ]:
method_results = load_method_results(data_dir)

if "zoh" in method_results:
    plot_method_results("zoh", method_results["zoh"], results_dir=None, show=True)

    # Trajectory plot
    sol_zoh = method_results["zoh"]["sol"]["exact_zoh_integrated"]
    hist_zoh = [h for h in method_results["zoh"]["history"] if h['method'] == 'exact_zoh_integrated']
    dts_zoh = np.array(hist_zoh[-1]['dts']).flatten()
    n_zoh = method_results["zoh"]["n"]
    plot_invpend_trajectory(sol_zoh, dts_zoh, n_zoh, title="ZOH: exact_zoh_integrated", show=True)

In [ ]:
method_results = load_method_results(data_dir)
loss_results = load_loss_results(data_dir, loss_names=selected_losses)
print_continuous_costs(method_results, loss_results)

## Alternative Loss Functions

ZOH base QP with alternative loss functions as regularizers alongside $\mathcal{L}_{\text{ocp}}$.

Combined loss: $\mathcal{L} = \mathcal{L}_{\text{ocp}} + \hat{\lambda} \cdot \mathcal{L}_{\text{reg}}$

where $\hat{\lambda}$ is adaptively tuned via gradient balancing.

In [ ]:
loss_results = load_loss_results(data_dir, loss_names=selected_losses)

for loss_name, result in loss_results.items():
    print(f"\n--- {loss_name} ---")
    plot_loss_results(loss_name, result, results_dir=None, show=True)

    dts_loss = np.array(result["history"][-1]['dts']).flatten()
    plot_invpend_trajectory(
        result["sol"], dts_loss, result["n"], title=loss_name, show=True
    )

In [ ]:
loss_results = load_loss_results(data_dir, loss_names=selected_losses)
plot_loss_comparison(loss_results, results_dir=None, show=True)

## Sampling Density and Trajectory Change Analysis

### Sampling Density vs Trajectory Changes

In [ ]:
method_results = load_method_results(data_dir)
loss_results = load_loss_results(data_dir, loss_names=selected_losses)
method_solutions = _build_method_solutions(method_results, loss_results)
plot_density_analysis(method_solutions, colors, results_dir=None, show=True)

### Cross-Correlation (Time-Lagged)

Cross correlation of the Sampling Density (SD) with:
- $\| \Delta u \|$
- $\| \Delta s \|_2$
- $\| u \|$
- $\| s \|_2$

Dotted lines at $\pm 1.96 / \sqrt{n}$ are the 95% CI. Outside bounds means statistically significant correlation.

- **lag = 0**: Instantaneous correlation
- **lag > 0**: Does high sampling density *precede* large changes?
- **lag < 0**: Does high sampling density *follow* large changes?

In [ ]:
method_results = load_method_results(data_dir)
loss_results = load_loss_results(data_dir, loss_names=selected_losses)
method_solutions = _build_method_solutions(method_results, loss_results)
plot_cross_correlation_analysis(method_solutions, colors, results_dir=None, show=True)

## Custom Training

Train with a fully configurable composite loss:
$$\mathcal{L} = \mathcal{L}_{\text{ocp}} + \sum_i w_i \cdot \mathcal{L}_i$$

**Settings:**
- `discretization`: `"zoh"` (exact ZOH via matrix exp) or `"euler"` (forward Euler)
- `loss_weights`: dict of `{loss_name: weight}`. Available losses:
  `L_IV`, `L_EQ`, `L_CPC`, `L_CSS`, `L_defect`, `L_dyn`, `L_equi`, `L_FI`, `L_SC`, `L_PWLH`

In [ ]:
from invpend_dt import train_custom_loss

# --- Configuration ---
loss_weights = {"L_SSD": 1000.0, "L_IV": 0.1}
discretization = "zoh"  # "zoh" or "euler"
n_steps = 40
n_epochs = 200
lr = 3e-2

In [ ]:
sol_custom, history_custom, n_custom = train_custom_loss(
    loss_weights, n=n_steps, n_epochs=n_epochs, lr=lr,
    discretization=discretization,
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)

epochs = [h["epoch"] for h in history_custom]

# Total loss
axes[0].plot(epochs, [h["loss"] for h in history_custom], label="total")
axes[0].plot(epochs, [h["loss_ocp"] for h in history_custom], label="L_ocp", ls="--")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss curves")
axes[0].legend(fontsize=7)

# Individual regularizer losses
for name in loss_weights:
    axes[1].plot(epochs, [h[f"loss_{name}"] for h in history_custom], label=name)
axes[1].set(xlabel="Epoch", ylabel="Loss", title="Regularizer losses")
axes[1].legend(fontsize=7)

# Final dt distribution
dts_final = np.array(history_custom[-1]["dts"]).flatten()
times_final = np.concatenate([[0], np.cumsum(dts_final)])
axes[2].step(times_final[:-1], dts_final, where="post")
axes[2].axhline(T / n_custom, color="gray", ls=":", label="uniform dt")
axes[2].set(xlabel="Time", ylabel="dt", title="Learned timesteps")
axes[2].legend(fontsize=7)

label = "L_ocp + " + " + ".join(f"{w}·{n}" for n, w in loss_weights.items())
fig.suptitle(f"Custom: {label}", fontsize=11)
plt.show()

In [ ]:
# Trajectory plot
dts_custom = np.array(history_custom[-1]["dts"]).flatten()
plot_invpend_trajectory(sol_custom, dts_custom, n_custom, title=f"Custom: {label}", show=True)